## Flow-Shop Scheduling Problem:
Projeto da Disciplina de Projeto e Análise de Algoritmos

### Importações

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import time

from fssp_branch_and_bound import FsspSolver

In [ ]:
with open('../data/instancias.json', 'r', encoding='utf-8') as arquivo:
    dados = json.load(arquivo)
data = pd.DataFrame(dados.get("instancias"))

print(dados.get("metadata"))
data.head()

### Filtragem:

In [ ]:
data["num_jobs"].value_counts()

In [ ]:
data_teste = data[data["num_jobs"] <= 11]

### Execução

In [ ]:
resultados = []

for indice, linha in data_teste.iterrows():
    for exec in range(5):
        nome = linha["nome_instancia"]
        num_trabalhos = linha["num_jobs"]
        num_maquinas = linha["num_machines"]
        tempos_de_trabalho = linha["matriz_tempos"]

        solver = FsspSolver(num_trabalhos, num_maquinas, tempos_de_trabalho)
        inicio = time.perf_counter()
        solucao, makespan = solver.run()
        fim = time.perf_counter()
    
        resultado = {
            "indice": indice,
            "nome_instancia": nome,
            "num_da_exec": exec,
            "num_tarefas": num_trabalhos,
            "num_maquinas": num_maquinas,
            "matriz_tempos": tempos_de_trabalho, 
            "solucao": solucao,
            "makespan": makespan,
            "tempo": fim-inicio,
        }
    
        resultados.append(resultado)

In [ ]:
resultados = pd.DataFrame(resultados)
resultados.head()

In [ ]:
resultados.to_csv("../data/resultados_fsspsolver.csv", index=False)

### Análise dos dados

In [ ]:
data = pd.read_csv("../data/resultados_fsspsolver.csv")
data.head()

In [ ]:
data_mean = data.groupby('num_tarefas')['tempo'].mean().reset_index()
plt.figure(figsize=(10, 6))
plt.plot(
    data_mean['num_tarefas'],
    data_mean['tempo']/60,
    marker='o',
    linestyle='-.',
    color='red',
    label='Tempo Médio de Execução'
)

x_exp = np.linspace(data_mean['num_tarefas'].min(), data_mean['num_tarefas'].max(), 100)
y_exp = 0.0000095 * np.exp(1,2 * x_exp)

plt.plot(
    x_exp,
    y_exp,
    linestyle='-',
    color='green',
    label='Tendência Exponencial'
)

plt.title('Relação entre Número de Tarefas e Tempo de Execução', fontsize=16)
plt.xlabel('Número de Tarefas', fontsize=12)
plt.ylabel('Tempo de Execução (minutos)', fontsize=12)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))

dados_boxplot = []
nomes_instancias = []

for instancia in data['nome_instancia'].unique():
    tempos = data[data['nome_instancia'] == instancia]['tempo']
    dados_boxplot.append(tempos)
    nomes_instancias.append(instancia)

plt.boxplot(dados_boxplot, labels=nomes_instancias)
plt.title('Distribuição do Tempo de Execução por Instância', fontsize=16)
plt.xlabel('Instância', fontsize=12)
plt.ylabel('Tempo de Execução (segundos)', fontsize=12)
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
tempo_medio_por_instancia = data.groupby('nome_instancia')['tempo'].mean().reset_index()
tempo_medio_por_instancia.columns = ['nome_instancia', 'tempo_medio_segundos']
tempo_medio_por_instancia['tempo_medio_minutos'] = tempo_medio_por_instancia['tempo_medio_segundos'] / 60

print("Tempo médio por instância:")
print(tempo_medio_por_instancia.round(2))